# Actividad: Dashboards y Plotly

*Analítica de datos y herramientas de inteligencia artificial
Grupo [601]*

* Jezrel Hernandez Alvarado - A01340173
* María Fernanda San Román Orozco - A01424691

**25/05/2026**

<img src="https://javier.rodriguez.org.mx/itesm/2014/tecnologico-de-monterrey-white.png" width="200">

# Steam Store Games

Contiene información de aprox 27,000 videojuegos de la plataforma Steam, recopilados utilizando las APIs de Steam y SteamSpy, tiene 18 columnas algunas son nombre del juego, fecha de lanzamiento, género, precio, número estimado de propietarios, reseñas, categorías y plataformas compatibles, entre otras. Permite análizar tendencias, como los géneros más populares, la relación entre precio y popularidad, el comportamiento de las reseñas de usuarios y la evolución de los juegos publicados en Steam. Además, contiene suficientes variables numéricas y categóricas para crear visualizaciones con Plotly, y de que no es sintetico ni generado artificialmente, ni lo hemos visto en clase con anterioridad.

**Referencia**: Davis, N. (2019). Steam Store Games (Clean dataset). Kaggle.Com. https://www.kaggle.com/datasets/nikdavis/steam-store-games/data
‌

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

px.defaults.height=400
px.defaults.width=800

In [2]:
df = pd.read_csv('steam.csv')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27075 entries, 0 to 27074
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   appid             27075 non-null  int64  
 1   name              27075 non-null  object 
 2   release_date      27075 non-null  object 
 3   english           27075 non-null  int64  
 4   developer         27074 non-null  object 
 5   publisher         27061 non-null  object 
 6   platforms         27075 non-null  object 
 7   required_age      27075 non-null  int64  
 8   categories        27075 non-null  object 
 9   genres            27075 non-null  object 
 10  steamspy_tags     27075 non-null  object 
 11  achievements      27075 non-null  int64  
 12  positive_ratings  27075 non-null  int64  
 13  negative_ratings  27075 non-null  int64  
 14  average_playtime  27075 non-null  int64  
 15  median_playtime   27075 non-null  int64  
 16  owners            27075 non-null  object

In [4]:
# Con el info podemos ver que no se cargó correctamente realease date, por lo que se convierte a datetime para poder hacer las gráficas correctamente si hay de tiempo
# Igual se creo otra columna de release year, con origen de release date.
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df['release_year'] = df['release_date'].dt.year

In [5]:
# Para gráficar más facilmente creamos otra columna del total de reseña, sumando el rating positivo con el negativo
# y despues calculamos el porcentaje de reseñas positivas
df['total_ratings'] = df['positive_ratings'] + df['negative_ratings']
df['rating_ratio'] = (df['positive_ratings'] / df['total_ratings'].replace(0,1) * 100).round(1)

In [6]:
# Como en géneros hay múltiples para cada juego, hacemos una columna con el principal nomas
# Y hacemos lo mismo con el de plataforma
df['main_genre'] = df['genres'].dropna().str.split(';').str[0].str.strip()
df['main_platform'] = df['platforms'].dropna().str.split(';').str[0].str.strip()

In [7]:
# Comprobamos que si se crearon todas las cokumnas y que no salten errores en el formato en head
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27075 entries, 0 to 27074
Data columns (total 23 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   appid             27075 non-null  int64         
 1   name              27075 non-null  object        
 2   release_date      27075 non-null  datetime64[ns]
 3   english           27075 non-null  int64         
 4   developer         27074 non-null  object        
 5   publisher         27061 non-null  object        
 6   platforms         27075 non-null  object        
 7   required_age      27075 non-null  int64         
 8   categories        27075 non-null  object        
 9   genres            27075 non-null  object        
 10  steamspy_tags     27075 non-null  object        
 11  achievements      27075 non-null  int64         
 12  positive_ratings  27075 non-null  int64         
 13  negative_ratings  27075 non-null  int64         
 14  average_playtime  2707

In [8]:
df.head(2)

,appid,name,release_date,english,developer,publisher,platforms,required_age,categories,genres,...,negative_ratings,average_playtime,median_playtime,owners,price,release_year,total_ratings,rating_ratio,main_genre,main_platform
0,10,Counter-Strike,2000-11-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,...,3339,17612,317,10000000-20000000,7.19,2000,127873,97.4,Action,windows
1,20,Team Fortress Classic,1999-04-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,...,633,277,62,5000000-10000000,3.99,1999,3951,84.0,Action,windows


# Esquemas de Color

Apartir de #2a475e que es el color del logo de Steam creamos en adobe color una paleta monocromatica

In [9]:
paleta = ['#2A475E','#5E7A91','#A7B7C4','#BADCF7','#8CCBFF']

# Gráficos con Plotly

### Histograma de Precios con Rug

In [10]:
px.histogram(
    df[df['price']<=90], x='price',
    color='main_platform',
    marginal='rug',
    color_discrete_sequence=[paleta[0],paleta[2],paleta[4]],
    title='Precios por plataforma',
    labels={'price':'Precio (USD)','main_platform':'Plataforma'},
    template='plotly_dark')

El histograma es de la distribución de precios de juegos de pago hasta 90 dolares, ya que el precio máximo en el dataset es 400, pero ya que hay muy pocos juegos arriba de 100, cuando haciamos el gráfico consideramos que quedaba muy grande y hacía dificil de interpretar con tantos bins. Mientras que el rug plot muestra la densidad real de puntos individuales.

Podemos ver que la mayoría cuesta menos de 10 dolares, siendo el pico más alto de entre 6 y 7 dolares, seguido de juegos entre 3 y 4 dolares, por lo que los juegos más frecuentes son los de precios accesibles. El color depende de las plataformas en las que este disponible, por lo que podemos ver que windows domina por mucho, y el rug permite que veamos la densidad más claramente, el color de template solo se utilizo en este gráfico ya que era para probar la función, pero consideramos que con este color en especifico es un poco más dificil ver los gráficos.

### Top 10 Géneros

In [11]:
top10 = df['genres'].dropna().str.split(';').explode().str.strip().value_counts().head(10).sort_values()

In [12]:
go.Figure(go.Bar(
    x=top10.values, y=top10.index, orientation='h',
    marker=dict(color=top10.values, colorscale=paleta, showscale=True),
    hovertemplate='Hay %{x:,} juegos del género %{y}<extra></extra>'))

La gráfica muestra los 10 géneros más frecuentes de juegos en Steam. Se separaron los géneros con split, y despúes se conto cada género por separado y se seleccionaron los top 10, y el hovertemplate se utilizó para mostrar la cantidad exacta de juegos.

Lo que se puede extraer del gráfico es que el genero más popuplar es indie con 19,421, seguido de action con 11 mil, y casual con 10 mil. Los más bajos del top 10 son free to play y sports, el color entre más claro es más juegos hay más de ese genero y entre más oscuro menos.


### Precio Vs Rating

In [13]:
px.scatter(df[df['price'].between(1,90) & (df['total_ratings']>=500)],
           x='price', y='rating_ratio', color='main_platform',
           color_discrete_sequence=[paleta[0],paleta[2],paleta[4]],
           title='Precio vs Ratings Positivos',
           labels={'price ':'Precio (USD) ','rating_ratio ':'% Positivos '})

Para este scatter filtramos precios entre 1 y 90 para quitar los juegos gratuitos y los outliers muy caros que aplastaban la escala y hacian mucho más dificil de interpretar la gráfica, y pusimos un mínimo de 500 ratings porque con muy pocos ratings el porcentaje no es representativo, un juego con 2 de 2 ratings positivos aparece como 100% aunque casi nadie lo haya jugado.

Se puede ver que no hay una relación clara entre precio y aprobación, ya que los juegos baratos y caros tienen niveles de aprobación similares. La mayoría de los puntos están entre 60% y 90% de aprobación sin importar el precio, y windows sigue dominando la cantidad de puntos.

### Top 10 Desarrolladores

In [14]:
top_dev = df['developer'].value_counts().head(10).sort_values()

go.Figure(go.Bar(
    x=top_dev.values, y=top_dev.index, orientation='h',
    marker_color=paleta[1],
    hovertemplate='<b>%{y}</b><br>%{x:,}<extra></extra>'))

Esta gráfica cuenta cuántos juegos tiene cada desarrollador y muestra los 10 con más títulos, solo tiene un color de la paleta porque no hay una variable categórica extra, y el hover muestra el número exacto.

Los desarrolladores con más juegos tienen entre 40 y 100 títulos, y tienenden a ser estudios pequeños que lanzan muchos juegos cortos o de bajo presupuesto, por ejemplo choices of games es el más grande de novelas interactivas, además esto confirma la gráfica anterior de lo más popular es indie.

### Tiempo de Juego vs Aprobación

In [15]:
px.scatter(df[df['average_playtime'].between(1,5000) & (df['total_ratings']>=100)],
           x='average_playtime', y='rating_ratio',
           color='main_platform',
           color_discrete_sequence=[paleta[0],paleta[2],paleta[4]],
           title='Tiempo de juego vs Aprobacion',
           labels={'average_playtime':'Tiempo prom (min)','rating_ratio':'% Positivos'})

Filtramos el tiempo de juego entre 1 y 5,000 minutos porque hay juegos con tiempos registrados altísimos e igual se veía raro, y el mínimo de 100 ratings es por la misma razón que en la gráfica anterior.

En cuanto a la interpretación se ve una relación fuerte entre cuánto tiempo se juega y qué tan bien calificado está, los juegos bien calificados aparecen en todos los rangos de tiempo, pero se puede ver un poco que los juegos con tiempos muy altos tienden a tener aprobaciones más estables de aprox 70 a 90%.

### Tiempo de juego por genero

In [16]:
top8 = df['main_genre'].value_counts().head(8).index

px.strip(df[df['main_genre'].isin(top8) & df['average_playtime'].between(1,3000)].sample(3000, random_state=42),
         x='main_genre', y='average_playtime', color='main_genre',
         hover_name='name', color_discrete_sequence=paleta,
         title='Tiempo de juego por genero', stripmode='overlay')

Como el dataset tiene 27,000 juegos, graficarlos todos hacía que la gráfica pesara mucho y se viera saturada, por eso tomamos una muestra de 3,000 con random_state para que sea reproducible. El rango de 1 a 3,000 minutos quita los ceros y los outliers.

Cada punto es un juego y se puede ver que la mayoría se agrupa abajo sin importar el género, pero los géneros como RPG y Strategy tienen más puntos dispersos hacia arriba, lo que tiene sentido porque son juegos más largos comparados con Casual o Indie.

### Lanzamientos por año y por plataforma

In [17]:
yr = df[df['release_year'].between(2000,2019)].groupby(['release_year','main_platform']).size().reset_index(name='n')

fig = go.Figure()
for plat, color in zip(['windows','mac','linux'], [paleta[0],paleta[2],paleta[4]]):
    s = yr[yr['main_platform']==plat]
    fig.add_trace(go.Scatter(x=s['release_year'], y=s['n'], name=plat.capitalize(),
                             mode='lines+markers', line=dict(color=color, width=3)))
fig

Agrupamos por año y plataforma y contamos cuántos juegos salieron en cada combinación. El rango de 2000 a 2019 es porque antes del 2000 hay muy pocos datos y después del 2019 el dataset ya no está completo.

Se ve claramente que Steam creció exponencialmente a partir de 2012, alcanzando su pico en 2018. Mac y Linux siguen una curva similar pero muy por debajo de Windows, lo que confirma que Windows es prácticamente el estándar.

### Tiempo promedio por genero

In [18]:
top8 = df['main_genre'].value_counts().head(8).index
avg  = df[df['main_genre'].isin(top8)].groupby('main_genre')['average_playtime'].mean().sort_values()

go.Figure(go.Bar(
    x=avg.values, y=avg.index, orientation='h',
    marker=dict(color=avg.values, colorscale=paleta, showscale=True),
    hovertemplate='<b>%{y}</b><br>%{x:.0f} min<extra></extra>'))

Calculamos el promedio de tiempo de juego para los 8 géneros más frecuentes. El color va de más oscuro a más claro según el tiempo, igual que en la gráfica de géneros, para mantener consistencia visual en el notebook.

RPG tiene el promedio más alto por mucho, lo que tiene sentido porque son juegos diseñados para jugarse durante decenas de horas. Casual y Indie tienen los promedios más bajos, lo que también es esperable dado que suelen ser experiencias más cortas.

### Ratings Positivos vs Negativos

In [19]:
px.scatter(df[(df['positive_ratings']<50000) & (df['negative_ratings']<10000)],
           x='negative_ratings', y='positive_ratings',
           color='main_platform',
           color_discrete_sequence=[paleta[0],paleta[2],paleta[4]],
           title='Ratings positivos vs negativos',
           labels={'positive_ratings':'Positivos','negative_ratings':'Negativos'})

Filtramos los positivos por debajo de 50,000 y los negativos por debajo de 10,000 porque hay algunos juegos muy famosos como CSGO o Dota que tienen millones de ratings y hacían que todos los demás juegos quedaran aplastados en la esquina inferior izquierda sin poder ver nada.

Podemos ver que los juegos con más ratings positivos también tienden a tener más negativos simplemente porque más gente los jugó y los calificó. Windows aparece con los juegos de mayor volumen de ratings, los de Mac y Linux se quedan más concentrados en los rangos bajos.

### Idioma por Plataforma


In [20]:
px.histogram(df[df['main_platform'].notna()],
             x='main_platform', color=df['english'].map({1:'Ingles',0:'Otro idioma'}),
             color_discrete_sequence=[paleta[0],paleta[3]],
             title='Idioma de juegos por plataforma',
             labels={'main_platform':'Plataforma','color':'Idioma'})

Esta gráfica es un conteo de cuántos juegos están en inglés versus otro idioma, agrupado por plataforma principal. Solo usamos dos colores de la paleta porque la variable solo tiene dos valores.

La gran mayoría de los juegos en todas las plataformas están en inglés, lo que refleja que el mercado angloparlante domina la distribución en Steam. La diferencia es proporcional en las tres plataformas, no parece haber una plataforma que tenga más diversidad de idiomas que otra.